# MLP Regressor para previsão de temperatura em Jena


## Base de dados

Base: `jena_climate_2009_2016.csv`

Série meteorológica de Jena, agregada de 10 minutos para frequência horária. O alvo é a temperatura do ar `T (degC)`.

- Granularidade: horária
- Random state: 42
- Divisão temporal: 80% inicial para treino e 20% final para teste


## Importações e configurações


In [ ]:
from copy import deepcopy
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox

import sys

PROJECT_ROOT = next(
    (pasta for pasta in (Path.cwd(), *Path.cwd().parents)
     if (pasta / 'validacao_bases.py').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Não foi possível localizar validacao_bases.py a partir do diretório atual.'
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from funcoes_series_temporais import (
    analisar_componente_residual, calcular_forca_sazonalidade,
    calcular_importancia_features,
)

In [ ]:
BASE_NAME = "clima"
SEED = 42
HORIZON = 1
SEASONAL_PERIOD = 24
EXOG_COLUMNS = ["p (mbar)", "rh (%)", "wv (m/s)", "wd_sin", "wd_cos"]
SAIDA = PROJECT_ROOT / 'resultados' / 'mlp'
SAIDA.mkdir(parents=True, exist_ok=True)
print(f"Base: {BASE_NAME} | resultados: {SAIDA}")

## Configuração do MLP Regressor


In [ ]:
# Configuração da MLP para previsão causal da próxima hora.
TARGET_LAGS = [1, 2, 3, 6, 12, 24, 48, 168]
EXOG_LAGS = {coluna: [1, 24] for coluna in EXOG_COLUMNS}
ROLLING_WINDOWS = [6, 24, 168]
SCALER = 'standard'
VALIDATION_RATIO = 0.15
PATIENCE = 20
MIN_DELTA = 1e-4
BATCH_SIZE = 128  # lote maior para manter a busca temporal executável
N_SPLITS_VALIDACAO = 3
MESES_VALIDACAO = 6
MAX_AMOSTRA_MAE_TREINO = 5000
CANDIDATOS_MLP = [
    {'nome': 'simples_32_relu_adam', 'hidden_layer_sizes': (32,), 'activation': 'relu', 'solver': 'adam', 'alpha': 1e-4, 'max_iter': 100},
    {'nome': 'media_64_relu_adam', 'hidden_layer_sizes': (64,), 'activation': 'relu', 'solver': 'adam', 'alpha': 1e-3, 'max_iter': 120},
    {'nome': 'aula_64x64_relu_adam', 'hidden_layer_sizes': (64, 64), 'activation': 'relu', 'solver': 'adam', 'alpha': 1e-4, 'max_iter': 100},
    {'nome': 'simples_32_tanh_adam', 'hidden_layer_sizes': (32,), 'activation': 'tanh', 'solver': 'adam', 'alpha': 1e-3, 'max_iter': 100},
    {'nome': 'simples_32_relu_sgd', 'hidden_layer_sizes': (32,), 'activation': 'relu', 'solver': 'sgd', 'alpha': 1e-4, 'max_iter': 120},
]

# 1. Preparação da base


## Base tratada

O notebook lê a base tratada com o corte temporal comum aos modelos. Os lags são criados antes da exclusão de exemplos incompletos.


In [ ]:
# Carrega a base tratada e verifica sua integridade.
from preparacao_bases import carregar_base_tratada

df, metadados_base = carregar_base_tratada(BASE_NAME, PROJECT_ROOT)
TARGET_COLUMN = metadados_base['alvo']
FREQUENCY = metadados_base['frequencia']
TRAIN_RATIO = metadados_base['treino']
SEED = metadados_base['seed']
FIM_TREINO = pd.Timestamp(metadados_base['fim_treino'])
INICIO_TESTE = pd.Timestamp(metadados_base['inicio_teste'])
np.random.seed(SEED)
diagnostico_base = pd.DataFrame([{
    'base': BASE_NAME, 'linhas': len(df), 'alvos_ausentes': int(df[TARGET_COLUMN].isna().sum()),
    'fim_treino': FIM_TREINO, 'inicio_teste': INICIO_TESTE, 'seed': SEED,
}])
display(diagnostico_base)

## Dicionário de dados

Identificação do alvo, das variáveis exógenas e de sua disponibilidade temporal para evitar vazamento de dados.


In [ ]:
# Registra o papel e a disponibilidade temporal das variáveis.
dicionario = pd.DataFrame([
    {
        'variavel': coluna,
        'papel': 'alvo' if coluna == TARGET_COLUMN else 'exogena',
        'tipo': str(df[coluna].dtype),
        'uso_no_modelo': 'somente valores defasados; calendário é conhecido antecipadamente',
    }
    for coluna in [TARGET_COLUMN] + EXOG_COLUMNS
]).set_index('variavel')
display(dicionario)

## Análise exploratória

Análise visual do alvo e das variáveis exógenas, com diagnóstico de valores ausentes e possíveis outliers.


In [ ]:
# Diagnóstico visual e sinais de outlier, sem remoção automática.
def diagnosticar_exploracao(dados, colunas, fator_iqr=1.5):
    faltantes = [coluna for coluna in colunas if coluna not in dados.columns]
    if faltantes:
        raise KeyError(f'Colunas ausentes na base: {faltantes}')
    linhas = []
    for coluna in colunas:
        serie = pd.to_numeric(dados[coluna], errors='coerce')
        validos = serie.dropna()
        q1, q3 = validos.quantile([0.25, 0.75])
        iqr = q3 - q1
        inferior, superior = q1 - fator_iqr * iqr, q3 + fator_iqr * iqr
        outliers = ((validos < inferior) | (validos > superior)).sum()
        linhas.append({
            'variavel': coluna, 'n': int(validos.size),
            'ausentes': int(serie.isna().sum()), 'media': float(validos.mean()),
            'desvio_padrao': float(validos.std()), 'minimo': float(validos.min()),
            'q1': float(q1), 'mediana': float(validos.median()), 'q3': float(q3),
            'maximo': float(validos.max()), 'limite_iqr_inferior': float(inferior),
            'limite_iqr_superior': float(superior), 'sinais_outlier_iqr': int(outliers),
            'percentual_outlier_iqr': float(outliers / max(1, validos.size)),
        })
    return pd.DataFrame(linhas).set_index('variavel')

colunas_t03 = [TARGET_COLUMN] + EXOG_COLUMNS
resumo_t03 = diagnosticar_exploracao(df, colunas_t03)
display(resumo_t03.round(3))

fig, axes = plt.subplots(len(colunas_t03), 1, figsize=(14, 12), sharex=True)
for ax, coluna in zip(axes, colunas_t03):
    df[coluna].resample('1D').mean().plot(ax=ax, linewidth=0.8, label=coluna)
    ax.set_ylabel(coluna)
    ax.legend(loc='upper right')
    ax.grid(alpha=0.25)
axes[-1].set_xlabel('Data')
fig.suptitle('Jena Climate: alvo e exógenas (médias diárias)', y=1.01)
plt.tight_layout()
plt.show()

print('Decisão: preservar valores extremos fisicamente plausíveis; não aplicar clipping global.')
print('Na modelagem, o alvo não é imputado; somente exógenas usam forward-fill causal de até 6 horas.')

# 2. Sazonalidade


## Decomposição STL

Decomposição da série de treino em tendência, sazonalidade e resíduos.


In [ ]:
# Seleciona o maior trecho horário contínuo para a decomposição.
def maior_trecho_continuo(serie):
    observada = serie.dropna().sort_index()
    blocos = observada.index.to_series().diff().ne(pd.Timedelta(hours=1)).cumsum()
    grupos = list(observada.groupby(blocos.to_numpy()))
    return max(grupos, key=lambda par: len(par[1]))[1].asfreq('h')

serie_stl = maior_trecho_continuo(df.loc[df.index < INICIO_TESTE, TARGET_COLUMN])
assert serie_stl.notna().all()
print(f'STL no treino: {serie_stl.index.min()} a {serie_stl.index.max()} ({len(serie_stl):,} horas)')
stl_resultado = calcular_forca_sazonalidade(serie_stl, periodo=SEASONAL_PERIOD)
decomposicao = stl_resultado['decomposicao']
print(f"Força sazonal diária: {stl_resultado['forca_sazonalidade']:.4f}")
print(f"Força de tendência: {stl_resultado['forca_tendencia']:.4f}")
fig = decomposicao.plot(); fig.set_size_inches(14, 8)
plt.tight_layout(); plt.show()

## Força da sazonalidade e resíduos

Cálculo da força sazonal e análise inicial do componente residual.


In [ ]:
# Analisa a sazonalidade e os resíduos da decomposição.
analise_residuos_stl = analisar_componente_residual(decomposicao, lags=24)
print(f"Média do residual: {analise_residuos_stl['media']:.6f}")
print(f"Desvio-padrão do residual: {analise_residuos_stl['desvio_padrao']:.6f}")
display(analise_residuos_stl["autocorrelacao_residual"])

# 3. Features e validação


## Pipeline de features

Construção de lags do alvo e das variáveis exógenas, janelas móveis e variáveis cíclicas de calendário.


In [ ]:
# Gera features causais na grade temporal completa.
from features_temporais import construir_features_temporais

model_data = construir_features_temporais(
    df, TARGET_COLUMN,
    lags_alvo=TARGET_LAGS,
    lags_exogenas=EXOG_LAGS,
    janelas_moveis=ROLLING_WINDOWS,
    horizonte=HORIZON, frequencia=FREQUENCY, remover_incompletos=True,
)

train = model_data.loc[model_data.index < INICIO_TESTE]
test = model_data.loc[model_data.index >= INICIO_TESTE]
if train.empty or test.empty or train.index.max() >= test.index.min():
    raise ValueError("Divisão temporal inválida.")
y_train, y_test = train["target"], test["target"]
X_train, X_test = train.drop(columns="target"), test.drop(columns="target")

print(f"Features: {X_train.shape[1]} | treino: {len(train):,} | teste final: {len(test):,}")
print(f"Corte comum: {FIM_TREINO} -> {INICIO_TESTE}")

## Escalonamento

O imputador e o escalonador são ajustados exclusivamente nos dados anteriores à janela de validação.


In [ ]:
# Ajusta o escalonamento apenas no trecho de treino.
def validar_X(X, nome):
    if not isinstance(X, pd.DataFrame) or X.empty:
        raise ValueError(f'{nome} deve ser um DataFrame não vazio')
    nao_numericas = X.select_dtypes(exclude='number').columns.tolist()
    if nao_numericas:
        raise TypeError(f'{nome} contém colunas não numéricas: {nao_numericas}')
    if np.isinf(X.to_numpy(dtype=float)).any():
        raise ValueError(f'{nome} contém valores infinitos')
    if isinstance(X.index, pd.DatetimeIndex) and not X.index.is_monotonic_increasing:
        raise ValueError(f'{nome} deve estar em ordem temporal crescente')

def criar_escalonador(tipo):
    if tipo == 'standard':
        return StandardScaler()
    if tipo == 'minmax':
        return MinMaxScaler()
    raise ValueError("tipo deve ser 'standard' ou 'minmax'")

n_validacao = max(24 * 30, int(len(X_train) * VALIDATION_RATIO))
X_ajuste, X_validacao = X_train.iloc[:-n_validacao], X_train.iloc[-n_validacao:]
y_ajuste, y_validacao = y_train.iloc[:-n_validacao], y_train.iloc[-n_validacao:]
preprocessador_validacao = Pipeline([
    ('imputador', SimpleImputer(strategy='median')),
    ('escalonador', criar_escalonador(SCALER)),
])
X_ajuste_sc = preprocessador_validacao.fit_transform(X_ajuste)
X_validacao_sc = preprocessador_validacao.transform(X_validacao)
escalonador_validacao = preprocessador_validacao.named_steps['escalonador']
if int(escalonador_validacao.n_samples_seen_) != len(X_ajuste):
    raise AssertionError('O escalonador consultou dados fora da janela de ajuste.')
diagnostico_escala = pd.DataFrame({
    'media_ajuste_escalado': X_ajuste_sc.mean(axis=0),
    'media_validacao_escalada': X_validacao_sc.mean(axis=0),
}, index=X_train.columns)
display(diagnostico_escala.head(10).round(4))
print('Escala validada: fit no passado; validação recebeu somente transform().')

## Validação temporal

São usados três folds cronológicos de seis meses. No teste final, o estimador e o escalonador permanecem congelados, e cada previsão utiliza somente informações disponíveis até a hora anterior.


In [ ]:
# Cria três folds cronológicos de seis meses.
fim_desenvolvimento = INICIO_TESTE
folds, linhas_folds = [], []
for i in range(N_SPLITS_VALIDACAO):
    inicio_val = fim_desenvolvimento - pd.DateOffset(
        months=MESES_VALIDACAO * (N_SPLITS_VALIDACAO - i)
    )
    fim_val = fim_desenvolvimento - pd.DateOffset(
        months=MESES_VALIDACAO * (N_SPLITS_VALIDACAO - i - 1)
    )
    ia = np.flatnonzero(X_train.index < inicio_val)
    iv = np.flatnonzero((X_train.index >= inicio_val) & (X_train.index < fim_val))
    if not len(ia) or not len(iv):
        raise ValueError('Fold temporal vazio.')
    assert X_train.index[ia].max() < X_train.index[iv].min() < INICIO_TESTE
    folds.append((ia, iv))
    linhas_folds.append({
        'fold': i + 1, 'treino_ate': X_train.index[ia].max(),
        'validacao_inicio': X_train.index[iv].min(),
        'validacao_fim': X_train.index[iv].max(),
        'n_treino': len(ia), 'n_validacao': len(iv),
    })
fold_table = pd.DataFrame(linhas_folds)
display(fold_table)
np.testing.assert_allclose(
    model_data['target_lag_1'], df[TARGET_COLUMN].shift(1).reindex(model_data.index)
)
print(f'Origens finais: {len(X_test):,} | horizonte: {HORIZON} hora | modelo congelado no teste.')

# 4. Modelagem MLP


## Arquitetura e otimização

Cinco configurações de camadas, ativação, solver e regularização são comparadas sem acessar o teste final. Cada candidata é avaliada nos três folds cronológicos; os 15% finais de cada treino controlam o early stopping.


In [ ]:
# Avalia as configurações com early stopping e validação temporal.
def treinar_com_early_stopping(configuracao, X_desenvolvimento, y_desenvolvimento):
    n_interno = max(24 * 30, int(len(X_desenvolvimento) * VALIDATION_RATIO))
    X_fit, X_val = X_desenvolvimento.iloc[:-n_interno], X_desenvolvimento.iloc[-n_interno:]
    y_fit, y_val = y_desenvolvimento.iloc[:-n_interno], y_desenvolvimento.iloc[-n_interno:]
    preprocessador = Pipeline([
        ('imputador', SimpleImputer(strategy='median')),
        ('escalonador', criar_escalonador(SCALER)),
    ])
    X_fit_sc = preprocessador.fit_transform(X_fit)
    X_val_sc = preprocessador.transform(X_val)
    modelo = MLPRegressor(
        hidden_layer_sizes=configuracao['hidden_layer_sizes'],
        activation=configuracao['activation'], solver=configuracao['solver'],
        alpha=configuracao['alpha'], batch_size=BATCH_SIZE, max_iter=1,
        warm_start=True, early_stopping=False, shuffle=True, random_state=SEED,
    )
    melhor_modelo, melhor_mae, melhor_epoca, sem_melhora = None, np.inf, 0, 0
    linhas = []
    inicio_amostra = max(0, len(X_fit_sc) - MAX_AMOSTRA_MAE_TREINO)
    for epoca in range(1, configuracao['max_iter'] + 1):
        modelo.partial_fit(X_fit_sc, y_fit)
        mae_treino = mean_absolute_error(
            y_fit.iloc[inicio_amostra:], modelo.predict(X_fit_sc[inicio_amostra:])
        )
        mae_validacao = mean_absolute_error(y_val, modelo.predict(X_val_sc))
        linhas.append({'epoca': epoca, 'MAE_treino': mae_treino, 'MAE_validacao': mae_validacao})
        if mae_validacao < melhor_mae - MIN_DELTA:
            melhor_modelo, melhor_mae, melhor_epoca = deepcopy(modelo), mae_validacao, epoca
            sem_melhora = 0
        else:
            sem_melhora += 1
            if sem_melhora >= PATIENCE:
                break
    return preprocessador, melhor_modelo, pd.DataFrame(linhas), melhor_epoca, float(melhor_mae)

inicio_busca = time.perf_counter()
resultados_folds = []
for configuracao in CANDIDATOS_MLP:
    for numero_fold, (ia, iv) in enumerate(folds, start=1):
        preprocessador, modelo, _, melhor_epoca, mae_interno = treinar_com_early_stopping(
            configuracao, X_train.iloc[ia], y_train.iloc[ia]
        )
        previsto_fold = modelo.predict(preprocessador.transform(X_train.iloc[iv]))
        resultados_folds.append({
            'nome': configuracao['nome'], 'fold': numero_fold,
            'MAE_validacao': mean_absolute_error(y_train.iloc[iv], previsto_fold),
            'melhor_epoca': melhor_epoca, 'MAE_early_stopping': mae_interno,
        })
        print(
            f"{configuracao['nome']} | fold {numero_fold}/{len(folds)} | "
            f"MAE={resultados_folds[-1]['MAE_validacao']:.4f} °C | época={melhor_epoca}",
            flush=True,
        )

df_folds = pd.DataFrame(resultados_folds)
tabela_busca = (df_folds.groupby('nome', as_index=False)
    .agg(MAE_validacao=('MAE_validacao', 'mean'),
         desvio_MAE=('MAE_validacao', 'std'),
         mediana_melhor_epoca=('melhor_epoca', 'median'))
    .sort_values('MAE_validacao').reset_index(drop=True))
melhor_nome = tabela_busca.loc[0, 'nome']
melhor_configuracao = next(
    configuracao.copy() for configuracao in CANDIDATOS_MLP
    if configuracao['nome'] == melhor_nome
)
_, _, historico_mlp, melhor_epoca_final, mae_interno_final = treinar_com_early_stopping(
    melhor_configuracao, X_train, y_train
)
melhor_configuracao['melhor_epoca'] = melhor_epoca_final
tempo_busca = time.perf_counter() - inicio_busca
display(tabela_busca.round(4))
display(df_folds.round(4))
print(f'Configuração selecionada: {melhor_nome} | busca: {tempo_busca / 60:.2f} min')

## Previsão fora da amostra

O modelo selecionado é reajustado no treino completo e avaliado no teste final com horizonte de uma hora. São calculados MAE, MSE, RMSE, R², sMAPE e resíduos.


In [ ]:
# Reajusta o modelo no treino completo e prevê o teste final.
def avaliar_previsoes(y, previsto, conjunto='teste'):
    residuo = (y - previsto).rename('residuo')
    mse = mean_squared_error(y, previsto)
    denominador = np.abs(y.to_numpy()) + np.abs(previsto.to_numpy())
    smape = np.mean(np.divide(
        2 * np.abs(y.to_numpy() - previsto.to_numpy()), denominador,
        out=np.zeros_like(denominador, dtype=float), where=denominador != 0
    )) * 100
    mape = (
        np.mean(np.abs((y.to_numpy() - previsto.to_numpy()) / y.to_numpy())) * 100
        if np.all(y.to_numpy() > 0) else np.nan
    )
    metricas = pd.DataFrame([{
        'conjunto': conjunto, 'n': len(y), 'MAE': mean_absolute_error(y, previsto),
        'MSE': mse, 'RMSE': mse ** 0.5, 'MAPE_percent': mape,
        'sMAPE_percent': smape, 'R2': r2_score(y, previsto),
        'vies_medio': residuo.mean(),
    }])
    return metricas, residuo

parametros_finais = {
    chave: melhor_configuracao[chave]
    for chave in ['hidden_layer_sizes', 'activation', 'solver', 'alpha']
}
modelo_mlp = Pipeline([
    ('imputador', SimpleImputer(strategy='median')),
    ('escalonador', criar_escalonador(SCALER)),
    ('mlp', MLPRegressor(
        **parametros_finais, batch_size=BATCH_SIZE,
        max_iter=max(1, int(melhor_configuracao['melhor_epoca'])),
        early_stopping=False, shuffle=True, random_state=SEED,
    )),
])
inicio_avaliacao = time.perf_counter()
modelo_mlp.fit(X_train, y_train)
previsao_treino = pd.Series(modelo_mlp.predict(X_train), index=y_train.index, name='previsao')
metricas_treino, residuos_treino = avaliar_previsoes(y_train, previsao_treino, 'treino')

previsao_mlp = pd.Series(modelo_mlp.predict(X_test), index=y_test.index, name='previsao')
metricas_mlp, residuos_mlp = avaliar_previsoes(y_test, previsao_mlp, 'teste_final')
tempo_avaliacao = time.perf_counter() - inicio_avaliacao
metricas_comparacao = pd.concat([metricas_treino, metricas_mlp], ignore_index=True)
display(metricas_comparacao.round(4))
print('MAPE fica NaN porque a temperatura contém valores zero/negativos; use sMAPE.')
wf = pd.DataFrame({
    'modelo': 'MLP Regressor', 'origem': X_test.index,
    'ultima_hora_observada': X_test.index - pd.Timedelta(hours=1),
    'fim_intervalo_previsto': X_test.index + pd.Timedelta(hours=1),
    'data': X_test.index, 'passo': HORIZON,
    'real': y_test.to_numpy(), 'previsto': previsao_mlp.to_numpy(),
    'residuo': residuos_mlp.to_numpy(),
    'persistencia': X_test['target_lag_1'].to_numpy(),
    'sazonal_24h': X_test['target_lag_24'].to_numpy(),
})
previsoes_mlp = wf.set_index('data')[['real', 'previsto', 'residuo']]
display(wf.head())

resumo_early_stopping = {
    'configuracao': melhor_nome,
    'epocas_executadas': len(historico_mlp),
    'melhor_epoca': int(melhor_configuracao['melhor_epoca']),
    'melhor_MAE_validacao': tabela_busca.loc[0, 'MAE_validacao'],
    'patience': PATIENCE,
    'tempo_busca_min': tempo_busca / 60,
    'tempo_avaliacao_min': tempo_avaliacao / 60,
}
display(pd.DataFrame([resumo_early_stopping]))
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(historico_mlp['epoca'], historico_mlp['MAE_treino'], label='treino')
ax.plot(historico_mlp['epoca'], historico_mlp['MAE_validacao'], label='validação temporal')
ax.axvline(resumo_early_stopping['melhor_epoca'], color='black', linestyle='--', label='melhor época')
ax.set(title='Evolução da loss (MAE)', xlabel='Época', ylabel='MAE (°C)')
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

df_folds.to_csv(SAIDA / 'folds_mlp_jena.csv', index=False)
tabela_busca.to_csv(SAIDA / 'grid_mlp_jena.csv', index=False)
wf.to_csv(SAIDA / 'previsoes_mlp_jena.csv', index=False)

## Síntese técnica

A MLP aproxima uma função não linear entre lags do alvo, medições meteorológicas passadas e ciclos de calendário. A camada final é linear porque a temperatura é contínua.

A preparação exige features causais e escala ajustada somente no treino. Os principais hiperparâmetros são camadas, neurônios, ativação, solver, regularização e número de épocas. A seleção usa validação temporal por MAE e early stopping interno.

Vantagens: relações não lineares, combinação flexível de variáveis e inferência rápida. Limitações: sensibilidade à escala e à inicialização, extrapolação fraca e menor interpretabilidade.


In [ ]:
# Resume as propriedades e os resultados do modelo selecionado.
print('Arquitetura selecionada:', melhor_configuracao['hidden_layer_sizes'])
print('Saída: linear, adequada à previsão contínua de temperatura.')
print('O sklearn otimiza erro quadrático; arquitetura e época são selecionadas por MAE temporal.')
print(f'Early stopping: patience={PATIENCE}; refit final usa a melhor época no treino completo.')
print('Teste: modelo congelado, com features causais atualizadas a cada origem, igual ao protocolo do RF.')
print('A MLP modela relações não lineares entre lags, exógenas e ciclos de calendário.')
print('Limitações: sensibilidade à escala, extrapolação fraca e dependência dos lags observados.')
display(metricas_comparacao.round(4))

# 5. Avaliação


## Comparação com baselines

Comparação fora da amostra entre o MLP, a persistência de uma hora e o baseline sazonal de 24 horas.


In [ ]:
# Compara o MLP com os baselines no teste final.
def resumir_metricas(nome, real, previsto):
    erro = np.asarray(real) - np.asarray(previsto)
    return {
        'modelo': nome, 'n': len(erro),
        'MAE': float(np.abs(erro).mean()),
        'RMSE': float(np.sqrt(np.mean(erro ** 2))),
        'R2': float(r2_score(real, previsto)),
        'vies_real_menos_previsto': float(erro.mean()),
        'P90_erro_absoluto': float(np.quantile(np.abs(erro), 0.9)),
        'MaxAE': float(np.abs(erro).max()),
    }

metricas = pd.DataFrame([
    resumir_metricas('MLP Regressor', wf['real'], wf['previsto']),
    resumir_metricas('Persistência (1h)', wf['real'], wf['persistencia']),
    resumir_metricas('Sazonal ingênuo (24h)', wf['real'], wf['sazonal_24h']),
])
mae_persistencia = metricas.loc[1, 'MAE']
metricas['ganho_MAE_vs_persistencia_pct'] = 100 * (1 - metricas['MAE'] / mae_persistencia)
display(metricas.round(4))
print(f'Busca: {tempo_busca / 60:.2f} min | treino e avaliação: {tempo_avaliacao / 60:.2f} min')
mensal = wf.set_index('data').resample('MS').apply({
    'residuo': lambda s: s.abs().mean(), 'real': 'count'
})
mensal.columns = ['MAE_MLP', 'n']
mensal['MAE_persistencia'] = (
    wf.set_index('data')['real'] - wf.set_index('data')['persistencia']
).abs().resample('MS').mean()
metricas.to_csv(SAIDA / 'metricas_mlp_jena.csv', index=False)
mensal.to_csv(SAIDA / 'mae_mensal_mlp_jena.csv')

## Diagnóstico dos resíduos

Análise dos erros por gráficos, função de autocorrelação e teste de Ljung–Box.


In [ ]:
# Diagnostica os resíduos fora da amostra.
residuos_mlp = wf.set_index('data')['residuo'].sort_index()
residuos_continuos = maior_trecho_continuo(residuos_mlp)
amostra_treino = previsao_treino.sample(n=min(5000, len(previsao_treino)), random_state=SEED).index
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.ravel()

axes[0].scatter(y_train.loc[amostra_treino], previsao_treino.loc[amostra_treino], alpha=0.25, s=10, label='treino')
axes[0].scatter(y_test, previsao_mlp, alpha=0.35, s=10, label='teste')
mn = min(y_train.min(), y_test.min(), previsao_treino.min(), previsao_mlp.min())
mx = max(y_train.max(), y_test.max(), previsao_treino.max(), previsao_mlp.max())
axes[0].plot([mn, mx], [mn, mx], '--', color='black', label='linha y=x')
axes[0].set(title='Real vs. previsto', xlabel='Temperatura real (°C)', ylabel='Temperatura prevista (°C)')
axes[0].legend()

axes[1].scatter(previsao_treino.loc[amostra_treino], residuos_treino.loc[amostra_treino], alpha=0.25, s=10, label='treino')
axes[1].scatter(previsao_mlp, residuos_mlp, alpha=0.35, s=10, label='teste')
axes[1].axhline(0, linestyle='--', color='black')
axes[1].set(title='Resíduos vs. previsto', xlabel='Temperatura prevista (°C)', ylabel='Resíduo: real - previsto (°C)')
axes[1].legend()

axes[2].plot(residuos_mlp.index, residuos_mlp, linewidth=0.6)
axes[2].axhline(0, linestyle='--', color='black')
axes[2].set(title='Resíduos fora da amostra ao longo do tempo', xlabel='Data', ylabel='Resíduo (°C)')

axes[3].hist(residuos_mlp, bins=30, density=True, alpha=0.8)
axes[3].axvline(0, linestyle='--', color='black')
axes[3].set(title='Histograma dos resíduos de teste', xlabel='Resíduo (°C)', ylabel='Densidade')
for ax in axes:
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(12, 4))
plot_acf(residuos_continuos, lags=168, zero=False, ax=ax)
ax.set_title('ACF dos resíduos — maior trecho horário contínuo, até 7 dias')
plt.tight_layout(); plt.show()

ljung_box_mlp = acorr_ljungbox(
    residuos_continuos, lags=[1, 6, 12, 24, 48, 168], return_df=True
)
ljung_box_mlp.index.name = 'lag_horas'
display(ljung_box_mlp)
if (ljung_box_mlp['lb_pvalue'] < 0.05).any():
    print('Há evidência de autocorrelação residual (p < 0,05); ainda resta estrutura temporal não capturada.')
else:
    print('Não há evidência de autocorrelação residual nos lags avaliados (p >= 0,05).')
ljung_box_mlp.to_csv(SAIDA / 'ljung_box_mlp_jena.csv')

## Importância das features

Importância por permutação, medida pelo aumento do MAE após embaralhar cada variável.


In [ ]:
# Estima a importância das features por permutação.
importancias_mlp = calcular_importancia_features(
    modelo_mlp, X_test, y_test, metodo='permutacao',
    n_repeticoes=10, seed=SEED, n_jobs=-1, max_amostras=3000,
)
display(importancias_mlp.head(15).round(4))
top = importancias_mlp.head(15).sort_values('importancia')
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top['feature'], top['importancia'], xerr=top['desvio_importancia'])
ax.set_title('Importância por permutação da MLP na origem do teste')
ax.set_xlabel('Aumento do MAE após permutação (°C)')
ax.grid(axis='x', alpha=0.25)
plt.tight_layout(); plt.show()
print('Principais features:', ', '.join(importancias_mlp.head(5)['feature']))
print('A importância mede o aumento do MAE ao embaralhar cada feature; valores maiores indicam maior contribuição preditiva.')
prefixos_exogenas = tuple(EXOG_COLUMNS)
importancias_exogenas = importancias_mlp[
    importancias_mlp['feature'].str.startswith(prefixos_exogenas)
].head(10)
display(importancias_exogenas.round(4))
print('Exógenas são medições passadas; nenhuma observação meteorológica futura foi usada.')
importancias_mlp.to_csv(SAIDA / 'importancia_mlp_jena.csv', index=False)
